In [ ]:
# Process ROIs if they're from combined side-by-side visualization
def process_combined_roi(roi_mask, target_shape):
    """Split side-by-side mask and transpose to match target shape."""
    if roi_mask.shape[0] == 1048:  # Combined side-by-side
        print(f"  Detected combined side-by-side ROI, splitting...")
        mid = 524
        left_half = roi_mask[:mid, :, :]
        right_half = roi_mask[mid:, :, :]
        
        # Choose half with more ROI data
        if np.sum(left_half > 0) > np.sum(right_half > 0):
            roi_mask = left_half
        else:
            roi_mask = right_half
        
        # Transpose to match image orientation
        if roi_mask.shape != target_shape:
            roi_mask = roi_mask.transpose(1, 0, 2)
    
    return roi_mask

print(f"Original B-mode shape: {bmode_data.shape}")

# Process ROIs to match original bmode shape
target_shape = (bmode_data.shape[0], bmode_data.shape[1], bmode_data.shape[2])
small_mc_roi_seg = process_combined_roi(small_mc_roi_seg, target_shape)
large_roi_seg = process_combined_roi(large_roi_seg, target_shape)

print(f"\nFinal shapes (all in Y, X, T format):")
print(f"  B-mode data: {bmode_data.shape}")
print(f"  Small MC ROI: {small_mc_roi_seg.shape}")
print(f"  Large ROI: {large_roi_seg.shape}")

n_frames = bmode_data.shape[2]
print(f"  Number of frames: {n_frames}")

# ============================================================================
# FRAME-BY-FRAME TEMPLATE MATCHING WITH CROSS-CORRELATION
# ============================================================================
from scipy.signal import correlate2d
from scipy.ndimage import gaussian_filter

print("\n" + "="*70)
print("FRAME-BY-FRAME TEMPLATE MATCHING")
print("="*70)

# STEP 1: Choose starting/reference frame
reference_frame = 150
print(f"\nStarting frame: {reference_frame}")

# STEP 2: Extract template region from starting frame using small ROI
mask_ref = small_mc_roi_seg[:, :, reference_frame] > 0
if np.sum(mask_ref) == 0:
    raise ValueError(f"No ROI found in reference frame {reference_frame}")

y_indices, x_indices = np.where(mask_ref)
template_padding = 5  # Small padding around ROI
y_min = max(0, y_indices.min() - template_padding)
y_max = min(bmode_data.shape[0], y_indices.max() + template_padding + 1)
x_min = max(0, x_indices.min() - template_padding)
x_max = min(bmode_data.shape[1], x_indices.max() + template_padding + 1)

print(f"\nTemplate region:")
print(f"  Bounding box: y=[{y_min}:{y_max}], x=[{x_min}:{x_max}]")
print(f"  Size: {y_max-y_min} x {x_max-x_min} pixels")

# Extract and preprocess template
template = bmode_data[y_min:y_max, x_min:x_max, reference_frame].copy()
template = gaussian_filter(template, sigma=1.0)  # Smooth for better correlation
template_normalized = (template - template.mean()) / (template.std() + 1e-10)

print(f"  Template stats: mean={template.mean():.3f}, std={template.std():.3f}")

# STEP 3: Define search parameters
search_margin = 20  # How many pixels to search around the original position
print(f"\nSearch parameters:")
print(f"  Search margin: ±{search_margin} pixels")

# STEP 4: Track template across all frames
print(f"\nTracking template across {n_frames} frames...")

shifts = np.zeros((n_frames, 2))  # (dy, dx)
correlation_scores = np.zeros(n_frames)
correlation_maps = []  # Store correlation maps for visualization

for t in range(n_frames):
    if t % 50 == 0:
        print(f"  Processing frame {t}/{n_frames}...")
    
    if t == reference_frame:
        shifts[t] = [0, 0]
        correlation_scores[t] = 1.0
        continue
    
    # Define search region in current frame
    search_y_min = max(0, y_min - search_margin)
    search_y_max = min(bmode_data.shape[0], y_max + search_margin)
    search_x_min = max(0, x_min - search_margin)
    search_x_max = min(bmode_data.shape[1], x_max + search_margin)
    
    # Extract and preprocess search region
    search_region = bmode_data[search_y_min:search_y_max, search_x_min:search_x_max, t].copy()
    search_region = gaussian_filter(search_region, sigma=1.0)
    search_region_normalized = (search_region - search_region.mean()) / (search_region.std() + 1e-10)
    
    # Compute normalized cross-correlation
    corr_map = correlate2d(search_region_normalized, template_normalized, mode='valid')
    corr_map = corr_map / template.size  # Normalize
    
    # Find peak correlation
    peak_idx = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    peak_corr = corr_map[peak_idx]
    
    # Convert peak location to displacement
    dy = peak_idx[0] + search_y_min - y_min
    dx = peak_idx[1] + search_x_min - x_min
    
    shifts[t] = [dy, dx]
    correlation_scores[t] = np.clip(peak_corr, 0, 1)
    
    # Store correlation map for key frames
    if t in [0, 50, 100, 200, 300, 500, 700, 900]:
        correlation_maps.append((t, corr_map, peak_idx))

print(f"\n✓ Template tracking complete!")
print(f"\nResults:")
print(f"  Displacement range:")
print(f"    Y: [{shifts[:, 0].min():.1f}, {shifts[:, 0].max():.1f}] pixels")
print(f"    X: [{shifts[:, 1].min():.1f}, {shifts[:, 1].max():.1f}] pixels")
print(f"  Mean displacement: {np.mean(np.linalg.norm(shifts, axis=1)):.2f} pixels")
print(f"  Correlation scores:")
print(f"    Mean: {correlation_scores.mean():.3f}")
print(f"    Min: {correlation_scores.min():.3f}")
print(f"    Max: {correlation_scores.max():.3f}")

# Show sample correlations
print(f"\nSample frames:")
for i in [0, 50, 100, 150, 200, 300, 500, 700, 900]:
    if i < n_frames:
        print(f"  Frame {i}: shift=({shifts[i, 0]:6.2f}, {shifts[i, 1]:6.2f}), correlation={correlation_scores[i]:.3f}")

# ============================================================================
# APPLY SHIFTS TO LARGE ROI
# ============================================================================
print("\n" + "="*70)
print("APPLYING SHIFTS TO LARGE ROI")
print("="*70)

large_roi_compensated = np.zeros_like(large_roi_seg)

for t in range(n_frames):
    if t % 100 == 0:
        print(f"  Applying shift to frame {t}/{n_frames}...")
    
    # Apply negative shift to align to reference
    shift_vector = -shifts[t]
    large_roi_compensated[:, :, t] = scipy_shift(
        large_roi_seg[:, :, t],
        shift_vector,
        order=0,
        mode='constant',
        cval=0
    )

print(f"\n✓ Motion compensation complete!")
print(f"  Output shape: {large_roi_compensated.shape} (Y, X, T)")

# Save
output_path = large_roi_path.replace('.nii', '_mc_template.nii')
nib.save(nib.Nifti1Image(large_roi_compensated.astype(np.uint8), np.eye(4)), output_path)
print(f"✓ Saved to: {output_path}")
print("="*70)

In [ ]:
# Visualize correlation scores and displacement vectors
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Correlation scores over time
axes[0, 0].plot(correlation_scores, linewidth=2)
axes[0, 0].axhline(0.3, color='r', linestyle='--', linewidth=2, label='Threshold (0.3)')
axes[0, 0].set_xlabel('Frame', fontsize=12)
axes[0, 0].set_ylabel('Correlation Score', fontsize=12)
axes[0, 0].set_title('Motion Tracking Quality Over Time', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Plot 2: Histogram of correlation scores
axes[0, 1].hist(correlation_scores, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(0.3, color='r', linestyle='--', linewidth=2, label='Threshold')
axes[0, 1].axvline(np.mean(correlation_scores), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(correlation_scores):.3f}')
axes[0, 1].set_xlabel('Correlation Score', fontsize=12)
axes[0, 1].set_ylabel('Number of Frames', fontsize=12)
axes[0, 1].set_title('Distribution of Correlation Scores', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Displacement vectors (Y and X)
axes[1, 0].plot(shifts[:, 0], label='Y displacement', linewidth=2, alpha=0.7)
axes[1, 0].plot(shifts[:, 1], label='X displacement', linewidth=2, alpha=0.7)
axes[1, 0].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].set_xlabel('Frame', fontsize=12)
axes[1, 0].set_ylabel('Displacement (pixels)', fontsize=12)
axes[1, 0].set_title('Displacement Vectors Over Time', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Displacement magnitude
displacement_magnitude = np.linalg.norm(shifts, axis=1)
axes[1, 1].plot(displacement_magnitude, color='purple', linewidth=2)
axes[1, 1].set_xlabel('Frame', fontsize=12)
axes[1, 1].set_ylabel('Displacement Magnitude (pixels)', fontsize=12)
axes[1, 1].set_title(f'Total Motion (Mean: {np.mean(displacement_magnitude):.2f} px)', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n{'='*70}")
print("CORRELATION SCORE STATISTICS:")
print(f"{'='*70}")
print(f"  Mean correlation: {np.mean(correlation_scores):.3f}")
print(f"  Median correlation: {np.median(correlation_scores):.3f}")
print(f"  Min correlation: {np.min(correlation_scores):.3f}")
print(f"  Max correlation: {np.max(correlation_scores):.3f}")
print(f"  Std deviation: {np.std(correlation_scores):.3f}")
print(f"\n  Frames with correlation >= 0.3: {np.sum(correlation_scores >= 0.3)} ({np.sum(correlation_scores >= 0.3)/len(correlation_scores)*100:.1f}%)")
print(f"  Frames with correlation >= 0.25: {np.sum(correlation_scores >= 0.25)} ({np.sum(correlation_scores >= 0.25)/len(correlation_scores)*100:.1f}%)")
print(f"  Frames with correlation < 0.25: {np.sum(correlation_scores < 0.25)} ({np.sum(correlation_scores < 0.25)/len(correlation_scores)*100:.1f}%)")
print(f"{'='*70}")

In [ ]:
# Helper: Identify good frames for T0 analysis based on correlation threshold
import matplotlib.pyplot as plt

# Set your correlation threshold
correlation_threshold = 0.6  # Adjust this based on your data

# Find frames with good tracking
good_frames = np.where(correlation_scores >= correlation_threshold)[0]
bad_frames = np.where(correlation_scores < correlation_threshold)[0]

print(f"Correlation threshold: {correlation_threshold}")
print(f"\nGood frames (correlation >= {correlation_threshold}): {len(good_frames)}/{n_frames} ({100*len(good_frames)/n_frames:.1f}%)")
print(f"Bad frames (correlation < {correlation_threshold}): {len(bad_frames)}/{n_frames} ({100*len(bad_frames)/n_frames:.1f}%)")

# Visualize which frames to keep/exclude
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(correlation_scores, linewidth=2, label='Correlation score')
ax.axhline(correlation_threshold, color='r', linestyle='--', linewidth=2, label=f'Threshold ({correlation_threshold})')
ax.fill_between(range(n_frames), 0, 1, where=(correlation_scores >= correlation_threshold), 
                 alpha=0.3, color='green', label='Good frames (use for T0)')
ax.fill_between(range(n_frames), 0, 1, where=(correlation_scores < correlation_threshold), 
                 alpha=0.3, color='red', label='Bad frames (exclude)')
ax.axvline(reference_frame, color='blue', linestyle=':', linewidth=2, label=f'Reference frame ({reference_frame})')
ax.set_xlabel('Frame', fontsize=12)
ax.set_ylabel('Correlation Score', fontsize=12)
ax.set_title('Frame Selection for T0 Analysis', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nGood frame numbers: {good_frames}")

# ============================================================================
# DIAGNOSTIC: CHECK CEUS DATA
# ============================================================================
print("\n" + "="*70)
print("DIAGNOSTIC: CHECKING CEUS DATA")
print("="*70)

# Load CEUS data if not already loaded
if 'ceus_data' not in locals():
    ceus_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p14/wk12/combined_nifti/p14_wk12_CHI_RAW.nii'
    print(f"\nLoading CEUS data from: {ceus_path}")
    ceus_data = nib.load(ceus_path).get_fdata()
    print(f"CEUS data shape: {ceus_data.shape}")

print(f"\nCEUS data statistics:")
print(f"  Overall min: {ceus_data.min():.6f}")
print(f"  Overall max: {ceus_data.max():.6f}")
print(f"  Overall mean: {ceus_data.mean():.6f}")
print(f"  Overall std: {ceus_data.std():.6f}")

# Get a sample pixel from the ROI
roi_mask = large_roi_compensated[:, :, reference_frame] > 0
roi_pixels = np.where(roi_mask)
sample_y, sample_x = roi_pixels[0][len(roi_pixels[0])//2], roi_pixels[1][len(roi_pixels[1])//2]

print(f"\nSample pixel at ({sample_y}, {sample_x}):")
sample_curve_all = ceus_data[sample_y, sample_x, :]
sample_curve_good = ceus_data[sample_y, sample_x, good_frames]

print(f"  All frames: min={sample_curve_all.min():.6f}, max={sample_curve_all.max():.6f}, mean={sample_curve_all.mean():.6f}")
print(f"  Good frames: min={sample_curve_good.min():.6f}, max={sample_curve_good.max():.6f}, mean={sample_curve_good.mean():.6f}")

# Plot sample pixel curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# All frames
axes[0].plot(sample_curve_all)
axes[0].axvline(reference_frame, color='r', linestyle='--', label='Reference frame')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Intensity')
axes[0].set_title(f'Sample Pixel - All Frames\n({sample_y}, {sample_x})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Good frames only
axes[1].plot(good_frames, sample_curve_good, 'g.-')
axes[1].set_xlabel('Frame number')
axes[1].set_ylabel('Intensity')
axes[1].set_title(f'Sample Pixel - Good Frames Only\n({len(good_frames)} frames)')
axes[1].grid(True, alpha=0.3)

# Show where peak is
axes[2].plot(sample_curve_all, alpha=0.5, label='All frames')
axes[2].plot(good_frames, sample_curve_good, 'g.-', label='Good frames')
peak_in_all = np.argmax(sample_curve_all)
peak_in_good = good_frames[np.argmax(sample_curve_good)]
axes[2].axvline(peak_in_all, color='orange', linestyle='--', label=f'Peak (all): frame {peak_in_all}')
axes[2].axvline(peak_in_good, color='green', linestyle='--', label=f'Peak (good): frame {peak_in_good}')
axes[2].set_xlabel('Frame')
axes[2].set_ylabel('Intensity')
axes[2].set_title('Peak Comparison')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print("ANALYSIS:")
print(f"{'='*70}")
if ceus_data.max() < 0.001:
    print("⚠️  WARNING: CEUS intensities are very low (< 0.001)")
    print("   This might not be contrast-enhanced data, or values need scaling")
elif sample_curve_all.max() - sample_curve_all.min() < 0.0001:
    print("⚠️  WARNING: Very little variation in pixel intensity over time")
    print("   No clear bolus enhancement detected")
else:
    print("✓  CEUS data looks reasonable")
    print(f"   Intensity range: {ceus_data.min():.6f} to {ceus_data.max():.6f}")

print(f"{'='*70}")